In [7]:
import pandas as pd, glob

total = 0
for arq in sorted(glob.glob("*.csv")):
    try:
        df = pd.read_csv(arq, sep=";", encoding="utf-8", low_memory=False)
    except Exception:
        df = pd.read_csv(arq, sep=";", encoding="latin-1", low_memory=False)
    print(arq, "->", len(df), "linhas,", len(df.columns), "colunas")
    total += len(df)

print("TOTAL:", total)

2020.csv -> 84920 linhas, 36 colunas
2021.csv -> 85147 linhas, 36 colunas
2022.csv -> 89659 linhas, 36 colunas
2023.csv -> 33874 linhas, 36 colunas
2024.csv -> 28972 linhas, 36 colunas
2025.csv -> 34797 linhas, 36 colunas
2026.csv -> 11524 linhas, 36 colunas
TOTAL: 368893


In [8]:
import pandas as pd, glob

dfs, colunas = [], []
for arq in sorted(glob.glob("*.csv")):
    try:
        df = pd.read_csv(arq, sep=";", encoding="utf-8", low_memory=False)
    except Exception:
        df = pd.read_csv(arq, sep=";", encoding="latin-1", low_memory=False)
    colunas.append(list(df.columns))
    df["arquivo_origem"] = arq
    dfs.append(df)

print("Colunas iguais em todos os anos:", all(c == colunas[0] for c in colunas))

bps = pd.concat(dfs, ignore_index=True)
print("Tamanho:", bps.shape)
print("Duplicados:", bps.drop(columns="arquivo_origem").duplicated().sum())
print("\nVazios por coluna:")
print(bps.isnull().sum())
print("\nTipos de dados:")
print(bps.dtypes)

Colunas iguais em todos os anos: True
Tamanho: (368893, 37)
Duplicados: 0

Vazios por coluna:
ano_compra                   0
cnpj_instituicao             0
sg_uf                        0
ds_esfera                    0
dt_compra                    0
dt_insercao               2142
validade_compra              0
co_catmat                    0
ds_item                      0
co_pdm                     401
co_grupo                   401
no_grupo                   401
co_classe                  401
no_classe                  401
fg_generico             180008
tp_compra                    0
sg_unidade_medida       234581
cnpj_fornecedor              0
no_fornecedor                0
cnpj_fabricante              0
no_fabricante                0
qt_medicamento               0
ds_observacao            59421
no_instituicao             312
no_municipio                 0
un_medida_capacidade        35
no_pdm                     401
nu_processo_compra           1
nu_ata                  278429
un_forn

In [9]:
print("Exemplos de data por ano:")
print(bps.groupby("arquivo_origem")["dt_compra"].first())

# Encoding: caracteres corrompidos
texto = bps.select_dtypes("object").columns
for c in texto:
    n = bps[c].astype(str).str.contains("Ã|Â|�", regex=True).sum()
    if n > 0:
        print("Possível encoding quebrado em", c, ":", n)

print("\nValores das colunas de filtro:")
for c in ["sg_uf", "ds_esfera", "tp_compra", "modalidade"]:
    print(bps[c].value_counts(dropna=False).head(12), "\n")

print("Resumo numérico:")
print(bps[["qt_medicamento", "vl_preco_unitario", "vl_preco_total"]].describe())
print("Preço <= 0:", (bps["vl_preco_total"] <= 0).sum())
print("co_seq_bps duplicado:", bps["co_seq_bps"].duplicated().sum())

Exemplos de data por ano:
arquivo_origem
2020.csv    21/10/2020
2021.csv    22/11/2021
2022.csv    04/01/2022
2023.csv    31/07/2023
2024.csv    15/01/2024
2025.csv    10/03/2025
2026.csv    09/01/2026
Name: dt_compra, dtype: object
Possível encoding quebrado em ds_item : 244920
Possível encoding quebrado em no_classe : 4193
Possível encoding quebrado em ds_observacao : 14186
Possível encoding quebrado em un_medida_capacidade : 459
Possível encoding quebrado em no_pdm : 13753
Possível encoding quebrado em nu_processo_compra : 7329
Possível encoding quebrado em nu_ata : 861
Possível encoding quebrado em un_fornecimento : 459

Valores das colunas de filtro:
sg_uf
PR    79413
SP    64733
RO    37424
RS    24089
PB    23432
PE    19540
SE    15570
PI    13510
PA    13083
AL    12407
MG    12179
SC    11503
Name: count, dtype: int64 

ds_esfera
MUNICIPAL    340728
ESTADUAL      28000
FEDERAL         122
PRIVADA          43
Name: count, dtype: int64 

tp_compra
ADMINISTRATIVA    362797
JUDIC

In [10]:
# 1) Encoding: só padrões realmente quebrados
padrao = r"Ã[\u0080-\u00BF]|Â[\u0080-\u00BF]|\ufffd"
for c in bps.select_dtypes("object").columns:
    achou = bps[c].astype(str).str.contains(padrao, regex=True)
    if achou.sum() > 0:
        print("Encoding quebrado em", c, ":", achou.sum())
        print(bps.loc[achou, c].head(3).tolist())
print("Teste de encoding concluído\n")

# 2) Maiores valores
cols = ["ano_compra","sg_uf","ds_item","qt_medicamento","vl_preco_unitario","vl_preco_total"]
print(bps.nlargest(10, "vl_preco_total")[cols].to_string())

# 3) O total confere com unitário x quantidade?
dif = (bps["vl_preco_total"] - bps["vl_preco_unitario"]*bps["qt_medicamento"]).abs()
print("\nLinhas em que o total difere de unitário x quantidade (>1%):", (dif > 0.01*bps["vl_preco_total"]).sum())

# 4) Peso dos maiores registros
total = bps["vl_preco_total"].sum()
print("Valor total geral:", round(total))
print("Parte do total nos 10 maiores registros: %.1f%%" % (100*bps["vl_preco_total"].nlargest(10).sum()/total))
print("Percentil 99,9% do valor total:", bps["vl_preco_total"].quantile(0.999))

Encoding quebrado em nu_processo_compra : 7329
['nÂ° 06/20', 'nÂ° 06/20', 'Ata nÂº 33/2020']
Teste de encoding concluído

        ano_compra sg_uf                                                                                                                                    ds_item  qt_medicamento  vl_preco_unitario  vl_preco_total
337089        2025    PR                                                                                                               PENICILAMINA, DOSAGEM:250 MG           77500          294400.00    2.281600e+10
348579        2025    SP                                                                        OCTREOTIDA, DOSAGEM:0,1 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL           28060          520000.00    1.459120e+10
363909        2026    PR                                                                                                            AMOXICILINA, CONCENTRAÇÃO:500MG          261600           51038.16    1.335158e+10
338123        2025

In [11]:
mediana = bps.groupby("co_catmat")["vl_preco_unitario"].transform("median")
qtd = bps.groupby("co_catmat")["vl_preco_unitario"].transform("count")
bps["razao_mediana"] = bps["vl_preco_unitario"] / mediana

total = bps["vl_preco_total"].sum()
for limite in [10, 50, 100, 1000]:
    sus = (bps["razao_mediana"] > limite) & (qtd >= 5)
    pct = 100 * bps.loc[sus, "vl_preco_total"].sum() / total
    print(f"Mais de {limite}x a mediana do item: {sus.sum()} registros, {pct:.1f}% do valor total")

Mais de 10x a mediana do item: 6016 registros, 57.5% do valor total
Mais de 50x a mediana do item: 1785 registros, 57.0% do valor total
Mais de 100x a mediana do item: 902 registros, 56.8% do valor total
Mais de 1000x a mediana do item: 121 registros, 51.1% do valor total


In [12]:
import numpy as np
import pandas as pd

# 1) Encoding do número do processo
bps["nu_processo_compra"] = bps["nu_processo_compra"].astype("string").str.replace("Â", "", regex=False)

# 2) Datas
bps["dt_compra"] = pd.to_datetime(bps["dt_compra"], format="%d/%m/%Y", errors="coerce")
bps["dt_insercao"] = pd.to_datetime(bps["dt_insercao"], dayfirst=True, format="mixed", errors="coerce")
bps["ano_mes"] = bps["dt_compra"].dt.strftime("%Y-%m")

# 3) CNPJs com 14 dígitos (recupera o zero à esquerda)
for c in ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]:
    bps[c] = bps[c].astype("string").str.zfill(14)

# 4) Códigos sem ".0"
for c in ["co_pdm", "co_grupo", "co_classe", "registro_anvisa"]:
    bps[c] = bps[c].round().astype("Int64")

# 5) Texto: tirar espaços e preencher vazios
for c in bps.select_dtypes(["object", "string"]).columns:
    bps[c] = bps[c].astype("string").str.strip().replace("", pd.NA)

vazios_texto = ["fg_generico", "sg_unidade_medida", "un_medida_capacidade", "un_fornecimento",
                "no_grupo", "no_classe", "no_pdm", "no_instituicao", "ds_observacao",
                "nu_ata", "nu_processo_compra"]
bps[vazios_texto] = bps[vazios_texto].fillna("Não informado")

# 6) Marcação de preços suspeitos (mais de 100x a mediana do item)
mediana = bps.groupby("co_catmat")["vl_preco_unitario"].transform("median")
qtd = bps.groupby("co_catmat")["vl_preco_unitario"].transform("count")
bps["razao_mediana"] = (bps["vl_preco_unitario"] / mediana).round(2)
bps["preco_suspeito"] = np.where((bps["razao_mediana"] > 100) & (qtd >= 5), "Sim", "Não")

bps = bps.drop(columns=["arquivo_origem"], errors="ignore")

# Conferências
print("Linhas:", len(bps), "(esperado 368893)")
print("dt_compra inválida:", bps["dt_compra"].isna().sum(), "(esperado 0)")
print("dt_insercao vazia:", bps["dt_insercao"].isna().sum(), "(esperado 2142)")
print("ano_compra diferente do ano da data:", (bps["ano_compra"] != bps["dt_compra"].dt.year).sum())
for c in ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]:
    print(c, "com tamanho diferente de 14:", (bps[c].str.len() != 14).sum())
print("Preços suspeitos:", (bps["preco_suspeito"] == "Sim").sum())
faltando = bps.isna().sum()
print("\nVazios restantes:")
print(faltando[faltando > 0])

Linhas: 368893 (esperado 368893)
dt_compra inválida: 0 (esperado 0)
dt_insercao vazia: 2142 (esperado 2142)
ano_compra diferente do ano da data: 0
cnpj_instituicao com tamanho diferente de 14: 0
cnpj_fornecedor com tamanho diferente de 14: 0
cnpj_fabricante com tamanho diferente de 14: 0
Preços suspeitos: 902

Vazios restantes:
dt_insercao          2142
co_pdm                401
co_grupo              401
co_classe             401
registro_anvisa    180008
vl_capacidade      234581
dtype: int64


In [13]:
import os
from google.colab import files

nome = "BPS_20_26_Jandir.csv"
bps.to_csv(nome, index=False, encoding="utf-8")
print("Tamanho do arquivo:", round(os.path.getsize(nome) / 1024**2), "MB")
files.download(nome)

Tamanho do arquivo: 218 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os, zipfile
from google.colab import files

nome = "BPS_20_26_Jandir"   # use o MESMO nome da vez anterior

# 1) Versão completa, compactada (GitHub)
bps.to_csv(nome + ".csv", index=False, encoding="utf-8")
with zipfile.ZipFile(nome + ".zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(nome + ".csv")
print("ZIP completo:", round(os.path.getsize(nome + ".zip") / 1024**2), "MB")

# 2) Versão enxuta (Looker Studio)
colunas = ["ano_compra", "dt_compra", "ano_mes", "sg_uf", "no_municipio", "ds_esfera",
           "no_instituicao", "cnpj_instituicao", "co_catmat", "ds_item", "no_classe",
           "tp_compra", "modalidade", "no_fornecedor", "no_fabricante",
           "un_fornecimento", "qt_medicamento", "vl_preco_unitario", "vl_preco_total",
           "razao_mediana", "preco_suspeito"]
bps[colunas].to_csv(nome + "_dashboard.csv", index=False, encoding="utf-8")
print("CSV do dashboard:", round(os.path.getsize(nome + "_dashboard.csv") / 1024**2), "MB")

files.download(nome + ".zip")
files.download(nome + "_dashboard.csv")

ZIP completo: 33 MB
CSV do dashboard: 129 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import os
from google.colab import files

nome = "BPS_20_26_Jandir"   # use o MESMO nome de antes

d = bps.copy()
d["item_resumo"] = d["ds_item"].astype("string").str.split(",").str[0].str.strip()
d["dt_compra"] = d["dt_compra"].dt.strftime("%Y%m%d")   # formato que o Looker reconhece como data

colunas = ["ano_compra", "dt_compra", "sg_uf", "no_municipio", "ds_esfera",
           "no_instituicao", "cnpj_instituicao", "co_catmat", "item_resumo", "no_classe",
           "tp_compra", "modalidade", "no_fornecedor", "cnpj_fornecedor", "no_fabricante",
           "un_fornecimento", "qt_medicamento", "vl_preco_unitario", "vl_preco_total",
           "preco_suspeito"]

arq = nome + "_dashboard.csv"
d[colunas].to_csv(arq, index=False, encoding="utf-8")
print("CSV do dashboard:", round(os.path.getsize(arq) / 1024**2, 1), "MB")
files.download(arq)

CSV do dashboard: 101.2 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import os
from google.colab import files

nome = "BPS_20_26_Jandir"   # use o MESMO nome de antes

d = bps.copy()
d["item_resumo"] = d["ds_item"].astype("string").str.split(",").str[0].str.strip()
d["dt_compra"] = d["dt_compra"].dt.strftime("%Y%m%d")
d["vl_preco_unitario"] = d["vl_preco_unitario"].round(4)
d["vl_preco_total"] = d["vl_preco_total"].round(4)

colunas = ["ano_compra", "dt_compra", "sg_uf", "no_municipio", "ds_esfera",
           "no_instituicao", "cnpj_instituicao", "co_catmat", "item_resumo", "no_classe",
           "tp_compra", "modalidade", "no_fornecedor", "cnpj_fornecedor", "no_fabricante",
           "un_fornecimento", "qt_medicamento", "vl_preco_unitario", "vl_preco_total",
           "preco_suspeito"]

arq = nome + "_dashboard.csv"
retiradas = []
for tirar in [None, "no_classe", "cnpj_instituicao", "un_fornecimento"]:
    if tirar:
        colunas.remove(tirar)
        retiradas.append(tirar)
    d[colunas].to_csv(arq, index=False, encoding="utf-8")
    mb = os.path.getsize(arq) / 1000**2
    print("Colunas retiradas:", retiradas, "->", round(mb, 1), "MB")
    if mb < 95:
        break

files.download(arq)

Colunas retiradas: [] -> 106.1 MB
Colunas retiradas: ['no_classe'] -> 94.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>